In [1]:
from IPython.core.display import HTML
table_css = 'table {align:left;display:block} '
HTML('<style>{}</style>'.format(table_css))

# 🎯 练习 1：面向 TinyML 的前向模式自动微分
## MAIE 5532：机器学习系统 - 第 2 周

### 学习目标：
- 理解前向模式 AD 中的双数表示法
- 实现用于传感器数据处理的前向模式 AD
- 使用数值微分验证 AD 结果
- 将理论与实际 TinyML 应用联系起来

### 🧠 什么是前向模式自动微分？

前向模式 AD 就像为你的数学计算配备了一个“数学助手”。它不是只计算 f(x)，而是在一次计算过程中同时计算 f(x) 和 f'(x)。

**为什么这很革命？**
- **精确导数：** 没有像有限差分那样的近似误差
- **高效：** 只需原函数大约两倍左右的计算成本
- **自动化：** 不需要手动推导复杂的导数公式
- **非常适合 TinyML：** 内存使用可预测，适合嵌入式系统

### 实际影响：
每当你使用机器学习——从语音助手到推荐系统——自动微分都在后台工作，计算让学习得以发生的梯度。

## 📚 数学基础：双数

### 什么是双数？
把双数想成“增强版”的数字，它保存两部分信息：

**双数 =（值，导数）**

数学上写作：a + b·ε，其中 ε² = 0

### 现实类比：
想象你在开车时，不仅追踪当前位置，还跟踪速度。位置是“值”，速度是“导数”（变化率）。

### 双数的运算规则：
- **加法：** (a₁, b₁) + (a₂, b₂) = (a₁ + a₂, b₁ + b₂)
- **乘法：** (a₁, b₁) × (a₂, b₂) = (a₁ × a₂, a₁ × b₂ + b₁ × a₂)
- **函数：** sin(a, b) = (sin(a), cos(a) × b)

### 关键见解：
这些规则会自动实现微积分中的链式法则！当我们乘两个双数时，我们实际上在自动应用乘积法则： (f × g)' = f' × g + f × g'

### 这为什么对机器学习重要：
不必手工推导复杂的导数公式（对拥有数百万参数的神经网络来说，这几乎是不可能的），我们让计算机自动应用这些简单规则来计算精确导数。

In [2]:
# 前向模式 AD 实现所需的基础导入
import math      # 用于 sin、cos 等数学函数
import numpy as np  # 用于数值运算（后续会用到验证）

# 用于清晰展示结果的工具函数
def show(title, *pairs):
    """
    用于格式化输出的辅助函数。

    这个函数让输出更易读，避免大量 print 混杂在一起。

    参数：
        title: 说明当前展示内容的标题
        *pairs: 需要显示的 (标签, 值) 对列表

    *pairs 语法表示：接受任意数量的参数。
    这让我们可以这样调用：
    show("标题", ("标签1", 值1), ("标签2", 值2), ...)
    """
    print(title)
    for k, v in pairs:
        print(f"  {k}: {v}")

print("✅ 导入成功!")
print()
print("📝 代码说明：")
print("   • math：提供 sin(), cos() 等基础数学函数")
print("   • numpy：后面会用于数值梯度检查")
print("   • show()：用于把输出格式化得更清楚")
print()
print("🐍 Python 概念 - *args：")
print("   *pairs 表示接受任意数量的参数")
print("   这使函数可以灵活处理多个输出项")
print("   就像餐馆菜单中说：‘想加什么配料都可以’")

✅ 导入成功!

📝 代码说明：
   • math：提供 sin(), cos() 等基础数学函数
   • numpy：后面会用于数值梯度检查
   • show()：用于把输出格式化得更清楚

🐍 Python 概念 - *args：
   *pairs 表示接受任意数量的参数
   这使函数可以灵活处理多个输出项
   就像餐馆菜单中说：‘想加什么配料都可以’


### 🔧 理解本节中使用的 Python 概念：

#### Python 魔术方法：
Python 有一些以双下划线开始和结束的方法，称为“魔术方法”或 “dunder 方法”：

- **__init__：** 构造函数，在创建新对象时被调用
- **__add__：** 当使用 + 运算符时调用
- **__mul__：** 当使用 * 运算符时调用
- **__repr__：** 当打印对象时调用

#### isinstance() 函数：
它用于检查某个对象是否属于某个特定类型。类似于问："这是双数还是普通数字？"

```python
isinstance(5, int)        # True - 5 是整数
isinstance(5.0, float)    # True - 5.0 是浮点数
isinstance(x, Dual)       # 如果 x 是 Dual 数，则返回 True
```

#### 为什么这很重要：
我们的 Dual 类需要同时兼容双数和普通数字。`isinstance()` 帮助我们自动处理这两种情况。

## 双数类完整代码：

In [3]:
class Dual:
    """
    用于前向模式自动微分的双数类。

    可以把它理解为一个“智能数字”，它记住两部分信息：
    - v：实际值（例如 2.5）
    - d：导数值（例如输入变量为 1.0，常数为 0.0）

    这正是前向模式 AD 的核心！
    """

    def __init__(self, v, d=0.0):
        """
        创建一个新的双数——这是构造函数。

        参数：
            v：数值（例如 2.5）
            d：导数值（例如输入变量为 1.0，常数为 0.0）

        示例：
            x = Dual(2.0, 1.0)  # 输入变量：x=2.0, dx/dx=1.0
            c = Dual(5.0, 0.0)  # 常数：c=5.0, dc/dx=0.0
        """
        self.v = float(v)
        self.d = float(d)
        print(f"Created dual number: value={self.v}, derivative={self.d}")

    def _wrap(self, other):
        """
        辅助函数：把普通数字转换为 Dual 数。

        这就像一个自动翻译器：
        - 如果 other 已经是 Dual，就直接返回
        - 如果 other 是普通数字，就转换为 Dual(other, 0.0)

        为什么常数的导数是 0.0？
        因为常数对自变量的导数为 0，例如 d/dx(5) = 0。
        """
        if isinstance(other, Dual):
            return other
        else:
            print(f"Converting regular number {other} to Dual({other}, 0.0)")
            return Dual(other, 0.0)

    def __add__(self, other):
        """
        双数加法： (a,b) + (c,d) = (a+c, b+d)

        微积分规则： (f + g)' = f' + g'
        """
        o = self._wrap(other)
        result = Dual(self.v + o.v, self.d + o.d)
        print(f"Addition: {self} + {o} = {result}")
        print(f"  Value: {self.v} + {o.v} = {result.v}")
        print(f"  Derivative: {self.d} + {o.d} = {result.d}")
        return result

    def __radd__(self, other):
        """
        反向加法：用于处理 5 + dual_number 这样的情况。

        Python 会先尝试 5.__add__(dual_number)
        由于普通数字不知道 Dual，所以这是失败的；
        然后 Python 会尝试 dual_number.__radd__(5)
        这时会调用这个方法。
        """
        return self.__add__(other)

    def __sub__(self, other):
        """
        减法： (a,b) - (c,d) = (a-c, b-d)

        微积分规则： (f - g)' = f' - g'
        """
        o = self._wrap(other)
        result = Dual(self.v - o.v, self.d - o.d)
        print(f"Subtraction: {self} - {o} = {result}")
        return result

    def __rsub__(self, other):
        """反向减法：处理 5 - dual_number 的情况"""
        o = self._wrap(other)
        result = Dual(o.v - self.v, o.d - self.d)
        print(f"Reverse subtraction: {other} - {self} = {result}")
        return result

    def __mul__(self, other):
        """
        乘法： (a,b) * (c,d) = (a*c, a*d + b*c)

        这就是微积分中的乘积法则！
        微积分规则： (f * g)' = f' * g + f * g'
        """
        o = self._wrap(other)
        result_value = self.v * o.v
        result_derivative = self.d * o.v + self.v * o.d
        result = Dual(result_value, result_derivative)

        print(f"Multiplication: {self} * {o} = {result}")
        print(f"  Value: {self.v} * {o.v} = {result_value}")
        print(f"  Derivative (Product Rule): {self.d} * {o.v} + {self.v} * {o.d} = {result_derivative}")
        print(f"  This automatically implements: (f*g)' = f'*g + f*g'")

        return result

    def __rmul__(self, other):
        """反向乘法：处理 5 * dual_number 的情况"""
        return self.__mul__(other)

    def __truediv__(self, other):
        """
        除法： (a,b) / (c,d) = (a/c, (b*c - a*d)/(c²))

        这就是微积分中的商法则！
        微积分规则： (f/g)' = (f'*g - f*g')/g²
        """
        o = self._wrap(other)
        result_value = self.v / o.v
        result_derivative = (self.d * o.v - self.v * o.d) / (o.v * o.v)
        result = Dual(result_value, result_derivative)

        print(f"Division: {self} / {o} = {result}")
        print(f"  Value: {self.v} / {o.v} = {result_value}")
        print(f"  Derivative (Quotient Rule): ({self.d} * {o.v} - {self.v} * {o.d}) / {o.v}² = {result_derivative}")

        return result

    def __rtruediv__(self, other):
        """反向除法：处理 5 / dual_number 的情况"""
        o = self._wrap(other)
        result_value = o.v / self.v
        result_derivative = (o.d * self.v - o.v * self.d) / (self.v * self.v)
        result = Dual(result_value, result_derivative)
        print(f"Reverse division: {other} / {self} = {result}")
        return result

    def __repr__(self):
        """
        字符串表示，便于调试与打印。

        当你打印 Dual 对象或在 Jupyter 中显示它时，就会调用这个方法。
        """
        return f"Dual(value={self.v}, derivative={self.d})"

print("✅ Dual 类已成功实现!")
print()
print("🧮 我们刚刚构建了什么：")
print("   • 一个“智能数字”，可同时保存值和导数")
print("   • 自动实现了微积分中的关键规则（乘积法则、商法则）")
print("   • 这是机器学习中精确计算导数的基础核心")
print()
print("🎯 关键见解：")
print("   每次我们对 Dual 进行加减乘除时，")
print("   都在自动应用微积分中的链式法则。")
print("   这就是自动微分的魔法！")

✅ Dual 类已成功实现!

🧮 我们刚刚构建了什么：
   • 一个“智能数字”，可同时保存值和导数
   • 自动实现了微积分中的关键规则（乘积法则、商法则）
   • 这是机器学习中精确计算导数的基础核心

🎯 关键见解：
   每次我们对 Dual 进行加减乘除时，
   都在自动应用微积分中的链式法则。
   这就是自动微分的魔法！


## 🔍 深入理解：我们刚刚构建了什么

### 乘积法则的魔法是如何实现的

当我们乘两个双数时，有件很神奇的事发生了。让我们拆开 `__mul__` 方法：

```python
def __mul__(self, other):
    result_value = self.v * o.v           # f(x) * g(x)
    result_derivative = self.d * o.v + self.v * o.d  # f'(x)*g(x) + f(x)*g'(x)
```

### 这就是微积分中的乘积法则！

### 为什么这很革命：
#### 在自动微分出现之前：
- 手工推导导数公式（容易出错且耗时）
- 使用有限差分法（只给近似值，速度慢）
- 只能处理相对简单的函数

#### 有了自动微分之后：
- 直接按 +、-、*、/ 来写函数
- 自动得到精确导数
- 适用任意复杂函数
- 可以扩展到拥有数百万参数的神经网络

#### 现实例子：
当 TensorFlow 或 PyTorch 训练神经网络时，本质上就在使用这些基础原则，只不过它被自动应用了很多次，并做得更高效、更复杂。

#### 内存效率：
每个 Dual 只需要存 2 个浮点数（值 + 导数）。这也是为什么前向模式 AD 非常适合嵌入式系统——内存需求稳定、可预测。

## 数学函数代码：

In [4]:
def sin(x):
    """
    双数版本的正弦函数。

    微积分规则：d/dx[sin(x)] = cos(x)

    这会自动实现链式法则：
    如果 x = Dual(value, derivative)，表示某个函数 g(x) 和 g'(x)
    那么 sin(x) = Dual(sin(value), cos(value) * derivative)

    这就给出 sin(g(x)) 及其导数 cos(g(x)) * g'(x)
    """
    if isinstance(x, Dual):
        result_value = math.sin(x.v)
        result_derivative = math.cos(x.v) * x.d
        result = Dual(result_value, result_derivative)

        print(f"sin({x}) = {result}")
        print(f"  Value: sin({x.v}) = {result_value}")
        print(f"  Derivative: cos({x.v}) * {x.d} = {result_derivative}")
        print(f"  Chain rule: d/dx[sin(g(x))] = cos(g(x)) * g'(x)")

        return result
    return math.sin(x)


def cos(x):
    """
    双数版本的余弦函数。

    微积分规则：d/dx[cos(x)] = -sin(x)
    注意这里有负号！
    """
    if isinstance(x, Dual):
        result_value = math.cos(x.v)
        result_derivative = -math.sin(x.v) * x.d
        result = Dual(result_value, result_derivative)

        print(f"cos({x}) = {result}")
        print(f"  Value: cos({x.v}) = {result_value}")
        print(f"  Derivative: -sin({x.v}) * {x.d} = {result_derivative}")

        return result
    return math.cos(x)


def relu(x):
    """
    双数版本的 ReLU 函数。

    ReLU(x) = max(0, x)

    导数规则：
    - 如果 x > 0：ReLU'(x) = 1
    - 如果 x ≤ 0：ReLU'(x) = 0

    这使 ReLU 成为“梯度门控”，在深度学习中非常重要。
    """
    if isinstance(x, Dual):
        if x.v > 0:
            result = Dual(x.v, x.d)
            print(f"ReLU({x}) = {result} (ACTIVE - gradient flows)")
            print(f"  Since {x.v} > 0, ReLU passes value and derivative through")
        else:
            result = Dual(0.0, 0.0)
            print(f"ReLU({x}) = {result} (INACTIVE - gradient blocked)")
            print(f"  Since {x.v} ≤ 0, ReLU outputs 0 for both value and derivative")

        return result

    return x if x > 0 else 0.0

print("✅ 数学函数已实现!")
print()
print("🧮 链式法则实现：")
print("   • sin(x)：自动应用 d/dx[sin(g(x))] = cos(g(x)) * g'(x)")
print("   • cos(x)：自动应用 d/dx[cos(g(x))] = -sin(g(x)) * g'(x)")
print("   • ReLU(x)：作为“梯度门控”，对深度学习极其关键")
print()
print("🎯 这意味着：")
print("   这些函数可以被用于更复杂的表达式中，")
print("   并且导数会自动正确地计算出来！")

✅ 数学函数已实现!

🧮 链式法则实现：
   • sin(x)：自动应用 d/dx[sin(g(x))] = cos(g(x)) * g'(x)
   • cos(x)：自动应用 d/dx[cos(g(x))] = -sin(g(x)) * g'(x)
   • ReLU(x)：作为“梯度门控”，对深度学习极其关键

🎯 这意味着：
   这些函数可以被用于更复杂的表达式中，
   并且导数会自动正确地计算出来！


## 📐 数学函数：链式法则在实践中

### 理解链式法则的实现

每个函数都会自动实现**链式法则**：

**链式法则：** 如果 y = f(g(x))，那么 dy/dx = f'(g(x)) × g'(x)

### 对于 sin(x)：
- 输入是一个 Dual(value, derivative)，表示 g(x) 和 g'(x)
- 输出是 Dual(sin(value), cos(value) × derivative)
- 这给出 sin(g(x)) 及其导数 cos(g(x)) × g'(x)

### 理解 ReLU 的特别之处

ReLU 在深度学习中非常重要，因为：

1. **计算效率高：** 只是简单的 max(0, x)
2. **梯度性质简单：** 要么通过（1），要么被阻断（0）
3. **解决梯度消失：** 对正值时不会饱和

### 为什么 ReLU 导数重要：
- **活跃神经元（x > 0）：** 梯度 = 1，学习继续
- **非活跃神经元（x ≤ 0）：** 梯度 = 0，神经元保持“关闭”

这会形成稀疏激活模式，有助于网络更有效地学习。

### 现实世界影响：
ReLU 的引入彻底改变了深度学习。它解决了深层网络训练中的梯度消失问题，使深度网络训练得以普及。

In [5]:
# 让我们用一个简单例子看看双数是如何工作的
print("=" * 60)
print("🎯 简单例子：f(x) = x² × sin(x)，在 x = 2.0 时")
print("=" * 60)
print()

print("这个例子展示了前向模式 AD 是如何逐步工作的。")
print("我们会同时计算函数值和它的导数。")
print()

# 第 1 步：创建输入变量
print("STEP 1: Create input variable")
print("-" * 30)
x = Dual(2.0, 1.0)  # x = 2.0, dx/dx = 1.0
print(f"Created input: {x}")
print(f"Interpretation: x = {x.v}, and since x is our input variable, dx/dx = {x.d}")
print()

# 第 2 步：计算 x²
print("STEP 2: Compute x²")
print("-" * 30)
print("We use our multiplication operator, which implements the product rule:")
a = x * x
print()

# 第 3 步：计算 sin(x)
print("STEP 3: Compute sin(x)")
print("-" * 30)
print("We use our sin function, which implements the chain rule:")
b = sin(x)
print()

# 第 4 步：计算最终结果
print("STEP 4: Compute x² × sin(x)")
print("-" * 30)
print("Final multiplication using the product rule:")
y = a * b
print()

print("=" * 60)
print("🎯 FINAL RESULT")
print("=" * 60)
show("f(x) = x² × sin(x) at x = 2.0",
     ("Function value f(2.0)", y.v),
     ("Derivative f'(2.0)", y.d))

print()
print("🔍 这意味着什么：")
print(f"   • 当 x=2 时，函数值为 {y.v:.6f}")
print(f"   • 当 x=2 时的斜率（导数）为 {y.d:.6f}")
print(f"   • 如果 x 只改变一个很小的量 δx，那么 f(x) 会约改变 {y.d:.6f} × δx")
print()
print("🎉 我们在不手工推导导数的前提下完成了求导！")
print("   链式法则和乘积法则都在 Dual 运算中自动应用。")

🎯 简单例子：f(x) = x² × sin(x)，在 x = 2.0 时

这个例子展示了前向模式 AD 是如何逐步工作的。
我们会同时计算函数值和它的导数。

STEP 1: Create input variable
------------------------------
Created dual number: value=2.0, derivative=1.0
Created input: Dual(value=2.0, derivative=1.0)
Interpretation: x = 2.0, and since x is our input variable, dx/dx = 1.0

STEP 2: Compute x²
------------------------------
We use our multiplication operator, which implements the product rule:
Created dual number: value=4.0, derivative=4.0
Multiplication: Dual(value=2.0, derivative=1.0) * Dual(value=2.0, derivative=1.0) = Dual(value=4.0, derivative=4.0)
  Value: 2.0 * 2.0 = 4.0
  Derivative (Product Rule): 1.0 * 2.0 + 2.0 * 1.0 = 4.0
  This automatically implements: (f*g)' = f'*g + f*g'

STEP 3: Compute sin(x)
------------------------------
We use our sin function, which implements the chain rule:
Created dual number: value=0.9092974268256817, derivative=-0.4161468365471424
sin(Dual(value=2.0, derivative=1.0)) = Dual(value=0.9092974268256817, derivativ

## 🎯 练习 1：TinyML 传感器数据处理

### 问题场景
**背景：** 你正在开发一个 TinyML 系统，用于环境监测。该系统会在把数据送入神经网络进行模式识别之前，对传感器读数进行处理。

### 处理流水线：
你的传感器预处理由三个步骤组成：

1. **归一化：** 把原始传感器读数从 [0, 1024] 映射到 [-1, 1] 范围
   - 公式： `(sensor_value - 512) / 512`
   - 原因：神经网络通常更适合处理归一化后的输入

2. **激活：** 应用 ReLU 去掉负值
   - 公式： `max(0, normalized_value)`
   - 原因：有时我们只关心正向偏差

3. **缩放：** 乘以 0.5 得到最终特征
   - 公式： `activated_value * 0.5`
   - 原因：让值保持在更合适的范围内

### 你的任务：
实现这个处理函数，并使用前向模式 AD 计算它的导数。

### 为什么需要导数？

#### 对边缘设备上的学习：
- **联邦学习：** 设备需要在本地计算梯度
- **持续学习：** 根据新数据模式适应预处理
- **迁移学习：** 针对具体环境微调预处理参数

#### 对分析：
- **灵敏度分析：** 输出对传感器噪声有多敏感？
- **鲁棒性：** 小的传感器误差会不会导致输出大幅变化？
- **特征工程：** 是否需要调整预处理参数？

#### 对系统设计：
- **梯度流：** 确保梯度能够回流到预处理模块
- **数值稳定性：** 避免梯度爆炸或消失
- **硬件优化：** 了解计算要求

In [6]:
def tinyml_feature_processing(s):
    """
    TinyML 传感器数据处理流水线，附逐步说明。

    这个函数代表嵌入式 ML 系统中常见的预处理步骤。
    我们会把传感器数据依次通过归一化、激活和缩放，再送入神经网络。

    流程：
    1. 归一化： (s - 512) / 512  [将 0-1024 映射到 -1 到 +1]
    2. 激活： ReLU(normalized)  [去掉负值]
    3. 缩放： result * 0.5      [最终缩放]

    参数：
        s：传感器读数（可以是普通数字或 Dual 数）

    返回：
        处理后的特征值（类型与输入类型一致）
    """
    s = s if isinstance(s, Dual) else Dual(float(s), 0.0)

    print("🔧 TINYML 传感器处理流水线")
    print("=" * 50)
    print(f"📊 输入传感器读数: {s}")
    print(f"   原始值: {s.v}")
    print(f"   输入导数: {s.d}（1.0 表示这是输入变量）")
    print()

    print("STEP 1: NORMALIZATION")
    print("-" * 25)
    print("🎯 目标：把 [0, 1024] 范围转成 [-1, 1]")
    print("📐 公式： (s - 512) / 512")
    print("💡 原因：神经网络通常更适合处理归一化后的输入")
    print()

    print("Sub-step 1a: Subtract offset (s - 512)")
    offset_removed = s - 512.0
    print(f"   {s.v} - 512 = {offset_removed.v}")
    print(f"   Derivative: d/ds[s - 512] = 1, so {s.d} - 0 = {offset_removed.d}")
    print()

    print("Sub-step 1b: Divide by scale factor ((s-512) / 512)")
    normalized = offset_removed / 512.0
    print(f"   {offset_removed.v} / 512 = {normalized.v}")
    print(f"   Derivative: d/ds[(s-512)/512] = 1/512 = {normalized.d}")
    print()

    print(f"✅ 归一化完成: {normalized}")
    print(f"   含义：传感器值 {s.v} → 归一化值 {normalized.v}")
    if normalized.v > 0:
        print(f"   该采样值位于中点（512）之上")
    else:
        print(f"   该采样值位于中点（512）之下")
    print()

    print("STEP 2: ReLU 激活")
    print("-" * 25)
    print("🎯 目标：去掉负值，保留正偏差")
    print("📐 公式：max(0, normalized_value)")
    print("💡 原因：有时我们只关心正向偏差")
    print()

    activated = relu(normalized)
    print(f"✅ ReLU 激活完成: {activated}")

    if activated.v > 0:
        print("   🟢 ReLU 生效：梯度可以继续传递")
        print(f"   梯度流动：{normalized.d} → {activated.d}")
    else:
        print("   🔴 ReLU 未生效：梯度被阻断（导数为 0）")
        print("   这意味着输入变化将不会影响输出")
    print()

    print("STEP 3: FINAL SCALING")
    print("-" * 25)
    print("🎯 目标：对激活值进行缩放，准备输入到神经网络")
    print("📐 公式：activated_value × 0.5")
    print("💡 原因：让值保持在更稳定的范围内")
    print()

    scaled = activated * 0.5
    print(f"✅ 缩放完成: {scaled}")
    print(f"   最终特征值: {scaled.v}")
    print(f"   灵敏度: {scaled.d}（表示输入每变动 1 个单位，输出变化多少）")
    print()

    print("🔍 结果解释：")
    print("-" * 30)
    print(f"• 原始传感器读数: {s.v}")
    print(f"• 处理后的特征值: {scaled.v}")
    print(f"• 灵敏度（导数）: {scaled.d}")
    print()
    print("灵敏度的含义：")
    if scaled.d > 0:
        print(f"  - 输入增加 1，会导致处理后特征增加 {scaled.d}")
        print(f"  - 传感器噪声 10 会导致约 {abs(scaled.d * 10):.6f} 的特征噪声")
    else:
        print("  - 输入变化不影响处理后的特征")
        print("  - 在该区域，系统对输入变化不敏感")
    print()

    return scaled

# 详细演示处理流程
print("🧪 测试处理流水线")
print("=" * 60)
print()

sensor_reading = 800.0
print(f"🎯 测试样例：传感器读数 = {sensor_reading}")
print(f"   这比中点（512）高出 {sensor_reading - 512} 个单位")
print(f"   预期：正归一化值 → ReLU 生效 → 导数继续传递")
print()

print("为了自动微分，创建 Dual 输入...")
sensor = Dual(sensor_reading, 1.0)
print(f"输入: {sensor}（导数=1.0 表示这是输入变量）")
print()

result = tinyml_feature_processing(sensor)

print("=" * 60)
print("🎉 最终结果")
print("=" * 60)
show("TinyML 特征处理结果",
     ("输入传感器读数", sensor.v),
     ("处理后的特征值", result.v),
     ("灵敏度（∂输出/∂输入）", result.d),
     ("每单位变化百分比", f"{result.d * 100:.6f}%"))

print()
print("🎯 关键洞察：")
print(f"• 原始传感器值 {sensor.v} 被处理为特征值 {result.v}")
print(f"• 系统灵敏度为 {result.d:.10f} = 1/1024")
print(f"• 这是合理的：归一化时除以 512，再乘以 0.5，等价于乘以 1/1024")
print(f"• 预处理过程稳定：输入小变化只会导致非常小的特征变化")

🧪 测试处理流水线

🎯 测试样例：传感器读数 = 800.0
   这比中点（512）高出 288.0 个单位
   预期：正归一化值 → ReLU 生效 → 导数继续传递

为了自动微分，创建 Dual 输入...
Created dual number: value=800.0, derivative=1.0
输入: Dual(value=800.0, derivative=1.0)（导数=1.0 表示这是输入变量）

🔧 TINYML 传感器处理流水线
📊 输入传感器读数: Dual(value=800.0, derivative=1.0)
   原始值: 800.0
   输入导数: 1.0（1.0 表示这是输入变量）

STEP 1: NORMALIZATION
-------------------------
🎯 目标：把 [0, 1024] 范围转成 [-1, 1]
📐 公式： (s - 512) / 512
💡 原因：神经网络通常更适合处理归一化后的输入

Sub-step 1a: Subtract offset (s - 512)
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number: value=288.0, derivative=1.0
Subtraction: Dual(value=800.0, derivative=1.0) - Dual(value=512.0, derivative=0.0) = Dual(value=288.0, derivative=1.0)
   800.0 - 512 = 288.0
   Derivative: d/ds[s - 512] = 1, so 1.0 - 0 = 1.0

Sub-step 1b: Divide by scale factor ((s-512) / 512)
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number

## 🧮 对结果的数学分析

### 理解导数值的含义

我们得到的导数值是 **0.0009765625**。它表示什么？

#### 数学分解：
0.0009765625 = 1/1024

**为什么恰好是 1/1024？**

让我们沿着整个流程追踪：
1. **归一化：** `(s - 512) / 512` → 导数 = `1/512`
2. **ReLU：** 对正数输入 → 导数 = `1`
3. **缩放：** `× 0.5` → 导数 = `0.5`

**合并起来：** `(1/512) × 1 × 0.5 = 1/1024`

### 工程解释：

#### 灵敏度分析：
- **1 个单位的传感器变化** → **0.001 的特征变化**
- **10 个单位的噪声** → **0.01 的特征噪声**
- **100 个单位的漂移** → **0.1 的特征变化**

#### 系统稳定性：
这个较低的灵敏度说明我们的预处理是相当稳健的：
- 小的传感器噪声不会显著影响模型输入
- 系统对环境干扰更稳定
- 在训练中，梯度不会爆炸

#### 训练时的梯度流：
如果我们做边缘端学习：
- 梯度能回流到预处理模块
- 1/1024 的因子意味着预处理梯度较小
- 这有助于避免整个系统中的梯度爆炸

In [7]:
    def numerical_gradient(f_scalar, x, epsilon=1e-6):
        """
        使用有限差分计算数值梯度。

        这是传统方法：
        f'(x) ≈ [f(x + ε) - f(x - ε)] / (2ε)

        这种方法：
        ✅ 适用于任意函数
        ❌ 只给出近似值，不是精确值
        ❌ 需要多次函数评估
        ❌ 对 ε 的选择很敏感
        ❌ 会受到数值精度影响

        参数：
            f_scalar：接收标量并返回标量的函数
            x：求导点
            epsilon：有限差分步长

        返回：
            数值近似导数
        """
        print(f"🔢 在 x = {x} 处计算数值梯度")
        print(f"   使用公式： [f(x+ε) - f(x-ε)] / (2ε)")
        print(f"   步长 ε = {epsilon}")

        f_plus = f_scalar(x + epsilon)
        f_minus = f_scalar(x - epsilon)

        print(f"   f({x + epsilon}) = {f_plus}")
        print(f"   f({x - epsilon}) = {f_minus}")

        gradient = (f_plus - f_minus) / (2 * epsilon)
        print(f"   梯度 ≈ ({f_plus} - {f_minus}) / (2 × {epsilon}) = {gradient}")

        return gradient

    def tinyml_scalar(s):
        """
        处理函数的标量版本，用于数值梯度验证。

        这个函数和 tinyml_feature_processing 的逻辑一致，
        但只使用普通 Python 数字，因为数值微分需要标量输入。
        """
        normalized = (float(s) - 512.0) / 512.0
        activated = normalized if normalized > 0 else 0.0
        scaled = activated * 0.5
        return scaled

    print("🔍 梯度验证")
    print("=" * 50)
    print()
    print("我们现在用传统数值微分方法来验证 AD 的结果。")
    print()
    print("这会展示：")
    print("✅ AD 结果与数值微分一致")
    print("✅ AD 更精确，因为它给出的是精确导数")
    print("✅ AD 更高效：只需一次前向计算")
    print()

    test_point = 800.0
    print(f"🎯 测试点：传感器读数 = {test_point}")
    print()

    print("方法 1：数值微分（传统）")
    print("-" * 55)
    numerical_grad = numerical_gradient(tinyml_scalar, test_point)
    print(f"   结果：{numerical_grad}")
    print()

    print("方法 2：自动微分（现代）")
    print("-" * 55)
    print("🤖 使用前向 AD 计算精确导数...")
    ad_input = Dual(test_point, 1.0)
    ad_result = tinyml_feature_processing(ad_input)
    print(f"   结果：{ad_result.d}")
    print()

    print("🔍 详细比较")
    print("=" * 30)
    error = abs(numerical_grad - ad_result.d)
    relative_error = error / abs(ad_result.d) if ad_result.d != 0 else float('inf')

    show("梯度比较",
         ("数值方法（近似）", f"{numerical_grad:.15f}"),
         ("自动微分（精确）", f"{ad_result.d:.15f}"),
         ("绝对误差", f"{error:.2e}"),
         ("相对误差", f"{relative_error:.2e}"))

    print()
    if error < 1e-5:
        print("✅ 验证通过!")
        print("   AD 结果与数值微分高度一致")
    else:
        print("❌ 验证失败!")
        print("   发现较大差异，应检查实现")

    print()
    print("🎯 为什么自动微分更优：")
    print("-" * 50)
    print("1. 🎯 精度：")
    print("   • AD 给出精确导数（仅受浮点数限制）")
    print("   • 数值方法只能给出近似导数")
    print()
    print("2. ⚡ 效率：")
    print("   • AD：一次前向计算")
    print("   • 数值法：至少两次函数计算（中心差分）")
    print("   • 对多个输入，数值法需要更多次计算")
    print()
    print("3. 🔒 稳定性：")
    print("   • AD 不依赖步长选择")
    print("   • 数值法会受到 ε 太大或太小的影响")
    print()
    print("4. 🚀 可扩展性：")
    print("   • AD 可以扩展到具有数百万参数的网络")
    print("   • 数值法难以在大规模问题上高效工作")
    print()
    print("这也是为什么现代机器学习框架都依赖自动微分!")

🔍 梯度验证

我们现在用传统数值微分方法来验证 AD 的结果。

这会展示：
✅ AD 结果与数值微分一致
✅ AD 更精确，因为它给出的是精确导数
✅ AD 更高效：只需一次前向计算

🎯 测试点：传感器读数 = 800.0

方法 1：数值微分（传统）
-------------------------------------------------------
🔢 在 x = 800.0 处计算数值梯度
   使用公式： [f(x+ε) - f(x-ε)] / (2ε)
   步长 ε = 1e-06
   f(800.000001) = 0.2812500009765625
   f(799.999999) = 0.2812499990234375
   梯度 ≈ (0.2812500009765625 - 0.2812499990234375) / (2 × 1e-06) = 0.0009765624975344167
   结果：0.0009765624975344167

方法 2：自动微分（现代）
-------------------------------------------------------
🤖 使用前向 AD 计算精确导数...
Created dual number: value=800.0, derivative=1.0
🔧 TINYML 传感器处理流水线
📊 输入传感器读数: Dual(value=800.0, derivative=1.0)
   原始值: 800.0
   输入导数: 1.0（1.0 表示这是输入变量）

STEP 1: NORMALIZATION
-------------------------
🎯 目标：把 [0, 1024] 范围转成 [-1, 1]
📐 公式： (s - 512) / 512
💡 原因：神经网络通常更适合处理归一化后的输入

Sub-step 1a: Subtract offset (s - 512)
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number: value=288.0, deriva

## 🛠️ 工程应用与现实场景

### TinyML 中哪些场景需要这种能力

#### 1. 边缘设备上的联邦学习
**场景：** 智能家居传感器学习用户的个性化模式
- **挑战：** 每台设备都必须在本地计算梯度
- **解决方案：** 前向模式 AD 使设备能够在本地计算梯度
- **影响：** 在不上传原始数据的前提下保留隐私

#### 2. 自适应传感器校准
**场景：** 恶劣环境中的环境监测传感器
- **挑战：** 传感器特性会随时间漂移
- **解决方案：** 导数告诉我们如何调整预处理参数
- **影响：** 实现自校准系统并维持长期准确性

#### 3. 实时异常检测
**场景：** 工业设备健康监测
- **挑战：** 需要理解不同传感器输入的敏感程度
- **解决方案：** 使用前向模式 AD 做实时灵敏度分析
- **影响：** 早期预警系统更容易解释，更容易优化

### 内存与计算分析

#### 内存需求（对嵌入式系统非常关键）：
传统神经网络推断：N 个参数 × 4 字节
- 前向模式 AD：N 个参数 × 8 字节（约 2 倍开销）
- 我们的预处理：3 个操作 × 8 字节 ≈ 24 字节

**对于 32KB 的嵌入式系统：** 我们的预处理仅占极小的一部分。

#### 计算需求：
原始预处理：3 个操作（减、除、乘）
- 前向模式 AD：总共约 6 个操作（约 2 倍开销）
- 执行时间：约为原始方法的 2 倍，但仍然可以做到实时

### 与其他方法的比较

#### 前向模式 vs 反向模式：
| 方面 | 前向模式 | 反向模式 |
|------|---------|---------|
| **内存** | O(1) - 常数 | O(graph) - 随计算图增长 |
| **最适合** | 少量输入，多输出 | 多输入，少输出 |
| **TinyML 适配度** | ✅ 非常适合 | ❌ 内存需求更大 |
| **实时性** | ✅ 可预测 | ❌ 受内存影响 |

#### 为什么前向模式适合预处理：
- **传感器处理：** 通常是 1 输入 → 1 输出，正适合前向模式
- **内存限制：** 嵌入式设备资源有限
- **实时要求：** 执行时间可预测
- **可解释性：** 容易理解每个输入的敏感度

In [8]:
print("🔬 高级分析：测试边界情况和系统行为")
print("=" * 70)
print()


def analyze_sensitivity_across_range():
    """
    在整个传感器范围内分析预处理行为，
    以帮助理解系统行为和潜在问题。
    """
    print("📊 传感器范围内灵敏度分析")
    print("-" * 45)
    print()

    test_points = [0, 256, 511, 512, 513, 768, 1024]

    print("传感器值 | 归一化值 | ReLU | 缩放值 | 导数 | 是否激活")
    print("--------|---------|------|--------|------|--------")

    for sensor_val in test_points:
        sensor = Dual(sensor_val, 1.0)
        normalized = (sensor - 512.0) / 512.0
        activated = Dual(normalized.v, normalized.d) if normalized.v > 0 else Dual(0.0, 0.0)
        scaled = activated * 0.5
        is_active = "Yes" if normalized.v > 0 else "No"
        print(f"{sensor_val:6d} | {normalized.v:10.3f} | {activated.v:4.3f} | {scaled.v:6.3f} | {scaled.d:10.6f} | {is_active}")

    print()
    print("🔍 关键观察：")
    print("• 当 ReLU 激活时，导数精确为 1/1024 = 0.000977")
    print("• 当传感器读数 ≤ 512 时，导数为 0（ReLU 未激活）")
    print("• 在传感器 = 512 处存在不连续点（ReLU 阈值）")
    print("• 在激活区域内系统是线性的，在非激活区域内为 0")


def test_extreme_cases():
    """测试极端情况下的行为"""
    print()
    print("⚠️ 极端情况测试")
    print("-" * 30)
    print()

    extreme_cases = [
        ("Very low sensor", 1.0),
        ("Just below threshold", 511.9),
        ("Just above threshold", 512.1),
        ("Very high sensor", 2000.0)
    ]

    for description, sensor_val in extreme_cases:
        print(f"🧪 {description}: {sensor_val}")
        sensor = Dual(sensor_val, 1.0)
        result = tinyml_scalar(sensor_val)

        if sensor_val > 512:
            derivative = 1.0 / 1024
            status = "Active"
        else:
            derivative = 0.0
            status = "Inactive"

        print(f"   Processed value: {result:.6f}")
        print(f"   Derivative: {derivative:.6f}")
        print(f"   ReLU status: {status}")
        print()


def demonstrate_gradient_flow():
    """展示梯度如何流过预处理流水线"""
    print("🔄 梯度流演示")
    print("-" * 35)
    print()
    print("这个例子展示了在更大的 ML 系统中，")
    print("梯度是如何从神经网络反向流回预处理步骤的。")
    print()

    upstream_gradient = 0.5
    sensor_val = 800.0

    print(f"📈 场景：上游梯度 = {upstream_gradient}")
    print(f"   （这通常来自神经网络训练中的反向传播）")
    print()

    sensor = Dual(sensor_val, 1.0)
    result = tinyml_scalar(sensor_val)
    local_gradient = 1.0 / 1024
    total_gradient = upstream_gradient * local_gradient

    print(f"🔗 链式法则应用：")
    print(f"   局部梯度（预处理）：{local_gradient:.6f}")
    print(f"   上游梯度（神经网络）：{upstream_gradient}")
    print(f"   总梯度：{upstream_gradient} × {local_gradient:.6f} = {total_gradient:.6f}")
    print()
    print(f"📊 解释：")
    print(f"   传感器读数的小变化会导致最终损失大约 {total_gradient:.6f} 的变化")
    print(f"   这就是训练过程中梯度如何指导学习的过程")

# 运行所有分析
analyze_sensitivity_across_range()
test_extreme_cases()
demonstrate_gradient_flow()

print("=" * 70)
print("🎯 高级分析总结")
print("=" * 70)
print("✅ 系统在整个传感器范围内都表现稳定")
print("✅ 梯度能够正确地流过预处理流水线")
print("✅ ReLU 在传感器 = 512 处形成清晰的激活/非激活边界")
print("✅ 该预处理流程适合边缘部署")

🔬 高级分析：测试边界情况和系统行为

📊 传感器范围内灵敏度分析
---------------------------------------------

传感器值 | 归一化值 | ReLU | 缩放值 | 导数 | 是否激活
--------|---------|------|--------|------|--------
Created dual number: value=0.0, derivative=1.0
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number: value=-512.0, derivative=1.0
Subtraction: Dual(value=0.0, derivative=1.0) - Dual(value=512.0, derivative=0.0) = Dual(value=-512.0, derivative=1.0)
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number: value=-1.0, derivative=0.001953125
Division: Dual(value=-512.0, derivative=1.0) / Dual(value=512.0, derivative=0.0) = Dual(value=-1.0, derivative=0.001953125)
  Value: -512.0 / 512.0 = -1.0
  Derivative (Quotient Rule): (1.0 * 512.0 - -512.0 * 0.0) / 512.0² = 0.001953125
Created dual number: value=0.0, derivative=0.0
Converting regular number 0.5 to Dual(0.5, 0.0)
Created dual number: value